First, we need to set up the environment with the necessary Hugging Face libraries, including bitsandbytes for memory-efficient model loading and llmlingua for the base token importance.

In [1]:
!pip install -q transformers datasets accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00


We will load the TweetEval dataset, the RoBERTa classifier, and the Qwen2.5 generator.

In [2]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSequenceClassification, BitsAndBytesConfig
)
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Dataset ───────────────────────────────────────────────────────────────
print("Loading dataset...")
dataset   = load_dataset("tweet_eval", "irony")
test_data = dataset["test"]

# ── RoBERTa irony classifier ──────────────────────────────────────────────
print("Loading RoBERTa classifier...")
roberta_name  = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model     = AutoModelForSequenceClassification.from_pretrained(
    roberta_name
).to(device)

# ── Qwen 2.5-3B (4-bit) ──────────────────────────────────────────────────
print("Loading Qwen2.5-3B (4-bit quantised)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
qwen_name      = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model     = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    device_map="auto",
    quantization_config=bnb_config
)
print("All models loaded.")

Device: cuda
Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

irony/train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

irony/test-00000-of-00001.parquet:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

All models loaded.


Preprocessing

In [3]:
import torch.nn.functional as F
import nltk, re
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

STOP_WORDS = set(stopwords.words('english'))

# ── Prompt template (used by both baselines) ──────────────────────────────
STRUCTURED_PROMPT = (
    "Analyze whether the following tweet is ironic or non-ironic. "
    "Think step-by-step, then end your response with EXACTLY one of:\n"
    "Verdict: IRONIC\n"
    "Verdict: NON-IRONIC\n\n"
    "Tweet: '{tweet}'"
)


def classify_tweet(tweet_text):
    """
    Classifies the raw tweet with RoBERTa.
    Returns (predicted_label, p_ironic).
    """
    inputs = rob_tokenizer(
        tweet_text, return_tensors="pt",
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        logits = rob_model(**inputs).logits
    probs      = F.softmax(logits, dim=-1).squeeze().cpu()
    p_ironic   = probs[1].item()
    prediction = int(p_ironic >= 0.5)
    return prediction, p_ironic


def vanilla_compress(cot_text, gamma=0.50):
    """
    Trims cot_text to (1 - gamma) of its original word count.
    Drops stopwords first, then shortest remaining words.
    No gradient sensitivity, no critical-token protection.
    Returns the compressed string.
    """
    words         = cot_text.split()
    target_length = max(5, int(len(words) * (1.0 - gamma)))

    scored = []
    for idx, word in enumerate(words):
        clean = word.lower().strip(".,!?\"'")
        score = 1 if (clean in STOP_WORDS or len(clean) < 3) else 10
        scored.append((idx, word, score))

    scored.sort(key=lambda x: x[2], reverse=True)
    kept = sorted(scored[:target_length], key=lambda x: x[0])
    return " ".join(w[1] for w in kept)


print("✅ Helpers ready: classify_tweet | vanilla_compress | STRUCTURED_PROMPT")

✅ Helpers ready: classify_tweet | vanilla_compress | STRUCTURED_PROMPT


Uncompressed CoT

In [4]:
import pandas as pd
from IPython.display import display

# ════════════════════════════════════════════════════════════════════════════
# BASELINE 1 — Uncompressed CoT  (γ = 0.00)
#
# What this measures:
#   • Accuracy  : RoBERTa on the raw tweet — no compression artefacts.
#   • Avg tokens: the full word count of every generated CoT.
#
# Classification is tweet-only (no CoT fusion) so this reflects the
# pure RoBERTa signal as a clean upper-bound reference point.
# ════════════════════════════════════════════════════════════════════════════

NUM_SAMPLES = 10

results_list        = []
correct_predictions = 0
total_cot_tokens    = 0

print("🚀 BASELINE 1 — Uncompressed CoT  (γ = 0.00)")
print(f"   N = {NUM_SAMPLES}\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # ── Generate full CoT ─────────────────────────────────────────────────
    prompt = STRUCTURED_PROMPT.format(tweet=sample_tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=120
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    cot_text      = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)
    cot_len       = len(cot_text.split())
    total_cot_tokens += cot_len

    # ── Classify original tweet ───────────────────────────────────────────
    prediction, p_ironic = classify_tweet(sample_tweet)

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  [{i+1:02d}] CoT={cot_len:3d} words | "
          f"P(ironic)={p_ironic:.3f} | Pred={prediction} | "
          f"True={true_label} | {'✅' if is_correct else '❌'}")

    results_list.append({
        "Tweet"           : sample_tweet[:40] + "...",
        "True"            : true_label,
        "Pred"            : prediction,
        "✓?"              : "✅" if is_correct else "❌",
        "P(ironic)"       : round(p_ironic, 3),
        "CoT Tokens"      : cot_len,
        "Tokens Removed"  : 0,
        "γ"               : 0.00,
    })

accuracy_pct   = (correct_predictions / NUM_SAMPLES) * 100
avg_cot_tokens = total_cot_tokens / NUM_SAMPLES

print(f"\n{'='*56}")
print(f"🏆 BASELINE 1 ACCURACY          : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📏 Avg CoT Tokens (uncompressed) : {avg_cot_tokens:.1f} words")
print(f"✂️  Tokens Removed               : 0  (no compression)")
print(f"{'='*56}")

df1 = pd.DataFrame(results_list)
display(df1)


🚀 BASELINE 1 — Uncompressed CoT  (γ = 0.00)
   N = 10

  [01] CoT=  7 words | P(ironic)=0.303 | Pred=0 | True=0 | ✅
  [02] CoT= 72 words | P(ironic)=0.650 | Pred=1 | True=1 | ✅
  [03] CoT= 94 words | P(ironic)=0.033 | Pred=0 | True=0 | ✅
  [04] CoT= 89 words | P(ironic)=0.209 | Pred=0 | True=0 | ✅
  [05] CoT= 78 words | P(ironic)=0.075 | Pred=0 | True=1 | ❌
  [06] CoT= 90 words | P(ironic)=0.981 | Pred=1 | True=0 | ❌
  [07] CoT= 74 words | P(ironic)=0.269 | Pred=0 | True=1 | ❌
  [08] CoT= 43 words | P(ironic)=0.090 | Pred=0 | True=0 | ✅
  [09] CoT= 90 words | P(ironic)=0.031 | Pred=0 | True=0 | ✅
  [10] CoT= 89 words | P(ironic)=0.866 | Pred=1 | True=1 | ✅

🏆 BASELINE 1 ACCURACY          : 70.0%  (7/10)
📏 Avg CoT Tokens (uncompressed) : 72.6 words
✂️  Tokens Removed               : 0  (no compression)


,Tweet,True,Pred,✓?,P(ironic),CoT Tokens,Tokens Removed,γ
0,@user Can U Help?||More conservatives ne...,0,0,✅,0.303,7,0,0.0
1,Just walked in to #Starbucks and asked f...,1,1,✅,0.650,72,0,0.0
2,#NOT GONNA WIN...,0,0,✅,0.033,94,0,0.0
3,@user He is exactly that sort of person....,0,0,✅,0.209,89,0,0.0
4,So much #sarcasm at work mate 10/10 #bor...,1,0,❌,0.075,78,0,0.0
5,Corny jokes are my absolute favorite...,0,1,❌,0.981,90,0,0.0
6,People complain about my backround pic a...,1,0,❌,0.269,74,0,0.0
7,"@user @user Darn, my sock joke needs fix...",0,0,✅,0.090,43,0,0.0
8,if Christian expects Fifa to sleep in my...,0,0,✅,0.031,90,0,0.0
9,"People who tell people with anxiety to ""...",1,1,✅,0.866,89,0,0.0


Vanilla TokenSkip

In [5]:
import pandas as pd
from IPython.display import display

# ════════════════════════════════════════════════════════════════════════════
# BASELINE 2 — Vanilla TokenSkip  (γ = 0.50, static)
#
# What this measures:
#   • Accuracy  : RoBERTa on the raw tweet after generating a CoT that
#                 is immediately discarded at 50% — no reasoning used.
#   • Avg tokens: compressed CoT word count (target = 50% of original).
#
# No entropy, no gradients, no verdict extraction.
# The same tweet is classified the same way every time regardless of
# how complex or ambiguous the content is — the core weakness RDS fixes.
# ════════════════════════════════════════════════════════════════════════════

VANILLA_GAMMA = 0.50
NUM_SAMPLES   = 10

results_list        = []
correct_predictions = 0
total_orig_tokens   = 0
total_comp_tokens   = 0

print(f"🚀 BASELINE 2 — Vanilla TokenSkip  (γ = {VANILLA_GAMMA}, static)")
print(f"   N = {NUM_SAMPLES}\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # ── Generate CoT ──────────────────────────────────────────────────────
    prompt = STRUCTURED_PROMPT.format(tweet=sample_tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=120
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    cot_text      = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)
    orig_len      = len(cot_text.split())
    total_orig_tokens += orig_len

    # ── Static 50% compression — no critical-token protection ─────────────
    compressed_text = vanilla_compress(cot_text, gamma=VANILLA_GAMMA)
    comp_len        = len(compressed_text.split())
    total_comp_tokens += comp_len
    tokens_removed  = orig_len - comp_len

    # ── Classify original tweet ───────────────────────────────────────────
    prediction, p_ironic = classify_tweet(sample_tweet)

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  [{i+1:02d}] Orig={orig_len:3d} → Comp={comp_len:3d} words "
          f"(removed {tokens_removed:2d}) | "
          f"P(ironic)={p_ironic:.3f} | Pred={prediction} | "
          f"True={true_label} | {'✅' if is_correct else '❌'}")

    results_list.append({
        "Tweet"           : sample_tweet[:40] + "...",
        "True"            : true_label,
        "Pred"            : prediction,
        "✓?"              : "✅" if is_correct else "❌",
        "P(ironic)"       : round(p_ironic, 3),
        "Orig Tokens"     : orig_len,
        "Comp Tokens"     : comp_len,
        "Tokens Removed"  : tokens_removed,
        "γ"               : VANILLA_GAMMA,
    })

accuracy_pct    = (correct_predictions / NUM_SAMPLES) * 100
avg_orig_tokens = total_orig_tokens / NUM_SAMPLES
avg_comp_tokens = total_comp_tokens / NUM_SAMPLES

print(f"\n{'='*56}")
print(f"🏆 BASELINE 2 ACCURACY             : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📏 Avg CoT Tokens — Original        : {avg_orig_tokens:.1f} words")
print(f"📏 Avg CoT Tokens — After Vanilla   : {avg_comp_tokens:.1f} words")
print(f"✂️  Avg Tokens Removed (static 50%)  : {avg_orig_tokens - avg_comp_tokens:.1f} words")
print(f"{'='*56}")

df2 = pd.DataFrame(results_list)
display(df2)

🚀 BASELINE 2 — Vanilla TokenSkip  (γ = 0.5, static)
   N = 10

  [01] Orig= 81 → Comp= 40 words (removed 41) | P(ironic)=0.303 | Pred=0 | True=0 | ✅
  [02] Orig=  6 → Comp=  5 words (removed  1) | P(ironic)=0.650 | Pred=1 | True=1 | ✅
  [03] Orig= 87 → Comp= 43 words (removed 44) | P(ironic)=0.033 | Pred=0 | True=0 | ✅
  [04] Orig= 95 → Comp= 47 words (removed 48) | P(ironic)=0.209 | Pred=0 | True=0 | ✅
  [05] Orig= 58 → Comp= 29 words (removed 29) | P(ironic)=0.075 | Pred=0 | True=1 | ❌
  [06] Orig= 93 → Comp= 46 words (removed 47) | P(ironic)=0.981 | Pred=1 | True=0 | ❌
  [07] Orig=102 → Comp= 51 words (removed 51) | P(ironic)=0.269 | Pred=0 | True=1 | ❌
  [08] Orig= 91 → Comp= 45 words (removed 46) | P(ironic)=0.090 | Pred=0 | True=0 | ✅
  [09] Orig= 91 → Comp= 45 words (removed 46) | P(ironic)=0.031 | Pred=0 | True=0 | ✅
  [10] Orig= 91 → Comp= 45 words (removed 46) | P(ironic)=0.866 | Pred=1 | True=1 | ✅

🏆 BASELINE 2 ACCURACY             : 70.0%  (7/10)
📏 Avg CoT Tokens — Origina

,Tweet,True,Pred,✓?,P(ironic),Orig Tokens,Comp Tokens,Tokens Removed,γ
0,@user Can U Help?||More conservatives ne...,0,0,✅,0.303,81,40,41,0.5
1,Just walked in to #Starbucks and asked f...,1,1,✅,0.650,6,5,1,0.5
2,#NOT GONNA WIN...,0,0,✅,0.033,87,43,44,0.5
3,@user He is exactly that sort of person....,0,0,✅,0.209,95,47,48,0.5
4,So much #sarcasm at work mate 10/10 #bor...,1,0,❌,0.075,58,29,29,0.5
5,Corny jokes are my absolute favorite...,0,1,❌,0.981,93,46,47,0.5
6,People complain about my backround pic a...,1,0,❌,0.269,102,51,51,0.5
7,"@user @user Darn, my sock joke needs fix...",0,0,✅,0.090,91,45,46,0.5
8,if Christian expects Fifa to sleep in my...,0,0,✅,0.031,91,45,46,0.5
9,"People who tell people with anxiety to ""...",1,1,✅,0.866,91,45,46,0.5
